In [4]:
import pandas as pd
from src.inference import translate_sentence, load_model
import torch
examples = [
    ("Мне пора идти спать.",
     "It's time for me to go to bed."),

    ("Мне пора идти спать",
     "It's time for me to go to bed."),

    ("мне пора идти спать.",
     "It's time for me to go to bed."),

    ("Я хочу увидеть результаты.",
     "I want to see the results."),

    ("Пожалуйста, закрой окно.",
     "Please close the window."),

    ("Где находится ближайшая аптека?",
     "Where is the nearest pharmacy?"),

    ("Почему ты не пришёл вчера?",
     "Why didn't you come yesterday?"),

    ("Сколько времени займёт эта работа?",
     "How long will this work take?"),

    ("Ты уже прочитал книгу, которую я тебе дал?",
     "Have you read the book I gave you yet?"),

    ("Кто оставил ключи на столе?",
     "Who left the keys on the table?"),

    ("Я не говорил, что он виноват.",
     "I didn't say he was guilty."),

    ("Я говорил, что он не виноват.",
     "I said he wasn't guilty."),

    ("Она никогда не была в Лондоне.",
     "She has never been to London."),

    ("Не открывай дверь, пока я не вернусь.",
     "Don't open the door until I get back."),

    ("Не все студенты сдали экзамен.",
     "Not all the students passed the exam."),

    ("Этот дом был построен в 1990 году.",
     "This house was built in 1990."),

    ("Поезд отправляется в 17:45.",
     "The train leaves at 5:45 p.m."),

    ("У меня было пять яблок, а осталось только два.",
     "I had five apples, but now I only have two left."),

    ("Билет стоит 250 рублей.",
     "The ticket costs 250 rubles."),

    ("Мой день рождения — 23 ноября.",
     "My birthday is on November 23."),

    ("Если завтра будет дождь, мы останемся дома.",
     "If it rains tomorrow, we'll stay home."),

    ("Когда я пришёл домой, она уже приготовила ужин.",
     "When I got home, she had already cooked dinner."),

    ("Книга, которую ты мне посоветовал, оказалась очень интересной.",
     "The book you recommended to me turned out to be very interesting."),

    ("Хотя он сильно устал, он продолжил работать.",
     "Although he was very tired, he kept working."),

    ("Я бы поехал с вами, если бы у меня было больше времени.",
     "I would go with you if I had more time."),

    ("У меня руки не доходят до этой работы.",
     "I haven't been able to get around to this work."),

    ("Мне всё равно, какой фильм мы будем смотреть.",
     "I don't mind which movie we watch."),

    ("Он взял себя в руки и продолжил выступление.",
     "He pulled himself together and continued his speech."),

    ("Ты меня неправильно понял.",
     "You misunderstood me."),

    ("На улице идёт сильный дождь.",
     "It's raining heavily outside."),
]

analysis = pd.DataFrame(examples, columns=["russian", "reference"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
bundle = load_model(
        name="bpe_attention_lstm",
        checkpoint_path="models/best_bpe_attention.pt",
        ru_model_path="artifacts/tokenizers/ru_bpe.model",
        en_model_path="artifacts/tokenizers/en_bpe.model",
        device=DEVICE
    )
analysis["prediction"] = analysis["russian"].apply(
    lambda text: translate_sentence(text, bundle)
)


with pd.option_context("display.max_colwidth", None):
    display(analysis)

,russian,reference,prediction
0,Мне пора идти спать.,It's time for me to go to bed.,It's time for me to go to bed.
1,Мне пора идти спать,It's time for me to go to bed.,I was time for go to bed.
2,мне пора идти спать.,It's time for me to go to bed.,It's time for me to go to bed.
3,Я хочу увидеть результаты.,I want to see the results.,I want to see results.
4,"Пожалуйста, закрой окно.",Please close the window.,Close the window.
5,Где находится ближайшая аптека?,Where is the nearest pharmacy?,Where is the nearest drugstore?
6,Почему ты не пришёл вчера?,Why didn't you come yesterday?,Why didn't you come yesterday?
7,Сколько времени займёт эта работа?,How long will this work take?,How long will this work take?
8,"Ты уже прочитал книгу, которую я тебе дал?",Have you read the book I gave you yet?,Have you read the book I gave gave you?
9,Кто оставил ключи на столе?,Who left the keys on the table?,Who left the keys keys?


In [5]:
reviews = [
    # 0
    ("сохранён", "корректно", "Корректный перевод"),

    # 1
    ("частично", "есть ошибки",
     "Неверное подлежащее I вместо It, необоснованное прошедшее время was "
     "и неправильная конструкция for go. Удаление точки ухудшило перевод"),

    # 2
    ("сохранён", "корректно", "Корректный перевод"),

    # 3
    ("сохранён", "корректно", "Корректный перевод"),

    # 4
    ("частично", "корректно",
     "Пропущено please: действие передано, но потеряна вежливая форма просьбы"),

    # 5
    ("сохранён", "корректно", "Корректный перевод"),

    # 6
    ("сохранён", "корректно", "Корректный перевод"),

    # 7
    ("сохранён", "корректно", "Корректный перевод"),

    # 8
    ("частично", "есть ошибки",
     "Повторено gave. После удаления повтора получится корректный перевод: "
     "Have you read the book I gave you?"),

    # 9
    ("частично", "есть ошибки",
     "Повторено keys и пропущено обстоятельство места 'на столе'"),

    # 10
    ("частично", "есть ошибки",
     "Начало I didn't say передано правильно, но придаточная часть разрушена: "
     "повторено was, а was his blame не передаёт 'он виноват'"),

    # 11
    ("частично", "корректно",
     "Местоимение he заменено на it: потеряно указание на человека мужского пола. "
     "Отрицание сохранено в нужной части предложения"),

    # 12
    ("сохранён", "корректно", "Корректный перевод"),

    # 13
    ("сохранён", "корректно", "Корректный перевод"),

    # 14
    ("сохранён", "корректно", "Корректный перевод"),

    # 15
    ("сохранён", "корректно", "Корректный перевод"),

    # 16
    ("частично", "есть ошибки",
     "Действие передано правильно, но время 17:45 искажено до 1:4454. "
     "Получена некорректная запись времени"),

    # 17
    ("частично", "есть ошибки",
     "Первая часть про пять яблок передана правильно. "
     "Во второй пропущено количество 'два', повторено left и нарушена конструкция"),

    # 18
    ("частично", "есть ошибки",
     "Сохранено число 250, но потеряна валюта 'рублей'. "
     "Вместо неё появился некорректный фрагмент 250s"),

    # 19
    ("частично", "есть ошибки",
     "Месяц и число распознаны, но число 23 продублировано. "
     "Отсутствие on само по себе здесь не является основной ошибкой"),

    # 20
    ("сохранён", "корректно", "Корректный перевод"),

    # 21
    ("частично", "корректно",
     "Got supper не передаёт точно 'приготовила ужин'. "
     "Потеряно 'уже' и указание, что приготовление завершилось до возвращения домой. "
     "Само слово supper допустимо"),

    # 22
    ("частично", "есть ошибки",
     "Неверно построена относительная часть the book you advised me you. "
     "Есть лишнее you и несогласованное you was. "
     "Также потерян оттенок 'оказалась'"),

    # 23
    ("частично", "корректно",
     "Основной смысл сохранён, но пропущено усиление 'сильно'. "
     "Continued working — корректная формулировка"),

    # 24
    ("сохранён", "корректно", "Корректный перевод"),

    # 25
    ("искажён", "есть ошибки",
     "Идиома 'руки не доходят' переведена буквально. "
     "Смысл 'не удаётся найти время или заняться работой' потерян; "
     "конструкция are not to work to this work некорректна"),

    # 26
    ("сохранён", "есть ошибки",
     "Общий смысл понятен, но в косвенном вопросе ошибочная инверсия "
     "shall we watch вместо прямого порядка слов. "
     "Естественнее: I don't care which movie we watch"),

    # 27
    ("частично", "корректно",
     "Идиома 'взял себя в руки' переведена буквально и не передаёт "
     "восстановление самообладания. Continued on не уточняет продолжение выступления. "
     "Структура грамматически допустима, но выражение неестественно"),

    # 28
    ("сохранён", "корректно", "Корректный перевод"),

    # 29
    ("частично", "есть ошибки",
     "Передан факт дождя на улице, но потеряна его интенсивность. "
     "Повтор raining outside делает предложение некорректным"),
]

assert len(reviews) == len(analysis)

analysis[["meaning", "grammar", "comment"]] = pd.DataFrame(
    reviews,
    columns=["meaning", "grammar", "comment"],
    index=analysis.index,
)

display(
    analysis.style
    .hide(axis="index")
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
    })
)

russian,reference,prediction,meaning,grammar,comment
Мне пора идти спать.,It's time for me to go to bed.,It's time for me to go to bed.,сохранён,корректно,Корректный перевод
Мне пора идти спать,It's time for me to go to bed.,I was time for go to bed.,частично,есть ошибки,"Неверное подлежащее I вместо It, необоснованное прошедшее время was и неправильная конструкция for go. Удаление точки ухудшило перевод"
мне пора идти спать.,It's time for me to go to bed.,It's time for me to go to bed.,сохранён,корректно,Корректный перевод
Я хочу увидеть результаты.,I want to see the results.,I want to see results.,сохранён,корректно,Корректный перевод
"Пожалуйста, закрой окно.",Please close the window.,Close the window.,частично,корректно,"Пропущено please: действие передано, но потеряна вежливая форма просьбы"
Где находится ближайшая аптека?,Where is the nearest pharmacy?,Where is the nearest drugstore?,сохранён,корректно,Корректный перевод
Почему ты не пришёл вчера?,Why didn't you come yesterday?,Why didn't you come yesterday?,сохранён,корректно,Корректный перевод
Сколько времени займёт эта работа?,How long will this work take?,How long will this work take?,сохранён,корректно,Корректный перевод
"Ты уже прочитал книгу, которую я тебе дал?",Have you read the book I gave you yet?,Have you read the book I gave gave you?,частично,есть ошибки,Повторено gave. После удаления повтора получится корректный перевод: Have you read the book I gave you?
Кто оставил ключи на столе?,Who left the keys on the table?,Who left the keys keys?,частично,есть ошибки,Повторено keys и пропущено обстоятельство места 'на столе'
